# LG HelloDoctor — 전체 파이프라인 통합
> A(STT) → B(LLM) → C(RAG+병원) → D(FastAPI)

## Step 1 — 라이브러리 설치

In [ ]:
!pip install unsloth chromadb sentence-transformers requests \
             fastapi uvicorn nest-asyncio pyngrok python-dotenv \
             beautifulsoup4 groq -q
print('설치 완료!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Step 2 — Google Drive 연결 + API 키

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

# API 키 (Colab Secrets에 저장)
KAKAO_API_KEY = userdata.get('KAKAO_API_KEY')
HIRA_API_KEY  = userdata.get('DATA_API_KEY')
GROQ_API_KEY  = userdata.get('GROQ_API_KEY')

print('Drive 연결 완료!')
print('API 키 로드 완료!')

## Step 3 — B팀 모델 로드

In [ ]:
import torch
import gc

# 찌꺼기 메모리 강제 수거 및 GPU 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!ls -l /content/drive/MyDrive/LGHellovision/LG_HelloDoctor/LLM/model

total 64357
-rw------- 1 root root     1213 Apr  7 12:04 adapter_config.json
-rw------- 1 root root 48679352 Apr  7 12:04 adapter_model.safetensors
-rw------- 1 root root     3827 Apr  7 12:04 chat_template.jinja
-rw------- 1 root root     5268 Apr  7 12:04 README.md
-rw------- 1 root root      427 Apr  7 12:04 tokenizer_config.json
-rw------- 1 root root 17209920 Apr  2 11:41 tokenizer.json


### Groq

In [ ]:
! pip install groq

In [ ]:
! pip install langchain_groq

In [ ]:
from langchain_groq import ChatGroq

# 이전에 정의된 GROQ_API_KEY를 사용하여 모델 초기화
# 대장이 선호하는 최신 성능의 llama-3.3-70b-versatile 모델을 기본으로 설정합니다.
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

print('✅ Groq API 기반 LLM 모델 연결 완료!')

#### local

In [ ]:
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/LLM/model',
    max_seq_length=2048,
    load_in_4bit=True
)

# attention_mask 경고 해결을 위해 pad_token 설정
tokenizer.padding_side = "right"
FastLanguageModel.for_inference(model)
print('✅ B팀 파인튜닝 모델 로드 완료!')

/tmp/ipykernel_12408/370496672.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
!pip install accelerate -U -q

In [ ]:
!ls /content/drive/MyDrive/whisper-ko-elderly

adapter_config.json	   checkpoint-300	     tokenizer_config.json
adapter_model.safetensors  preprocessor_config.json  tokenizer.json
checkpoint-100		   processor_config.json     training_args.bin
checkpoint-200		   README.md


## Step 3-1 A Whisper 모델 고도화

In [ ]:
import torch
import gc

# 찌꺼기 메모리 강제 수거 및 GPU 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# 변수 이름을 model -> stt_model로 변경하여 충돌 방지
stt_model = WhisperForConditionalGeneration.from_pretrained(
    '/content/drive/MyDrive/whisper-ko-elderly'
).to("cuda").float()

stt_processor = WhisperProcessor.from_pretrained(
    '/content/drive/MyDrive/whisper-ko-elderly'
)
print('✅ STT 전 전용 모델(stt_model) 로드 완료!')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [ ]:
# 1. 파이썬 패키지 설치
!pip install sounddevice soundfile openai-whisper

# 2. 리눅스 시스템 라이브러리 설치 (PortAudio 의존성 해결)
!apt-get install libportaudio2

In [ ]:
# =============================================================================
# Step 3-2. [A팀] Wake Word 감지 모듈 (Step 0)
# =============================================================================
import tempfile
import numpy as np
import sounddevice as sd
import soundfile as sf
import os

# 필수 라이브러리 설치 안내 (실행 시 주석 해제)
# !pip install sounddevice soundfile openai-whisper

class WakeWordDetector:
    def __init__(self, wake_words=None, use_whisper=True):
        self.wake_words = wake_words or ["헬로비", "헬로 비", "헬로비비", "헬로 비비"]
        self.use_whisper = use_whisper
        self._model = None
        if use_whisper:
            self._load_tiny_whisper()

    def _load_tiny_whisper(self):
        try:
            import whisper as _whisper
            # GPU 메모리 절약을 위해 cpu로 로드하거나 필요할 때만 호출 권장
            # 여기서는 B팀 모델과 충돌을 피하기 위해 CPU 로드 설정을 권장합니다.
            self._model = _whisper.load_model("tiny", device="cpu")
            print("✅ [Wake word] Whisper-tiny(CPU) 로드 완료")
        except Exception as e:
            print(f"❌ [Wake word] 로드 실패: {e}")
            self.use_whisper = False

    def _is_speech(self, audio):
        ENERGY_THRESHOLD = 0.008
        rms = float(np.sqrt(np.mean(audio ** 2)))
        return rms > ENERGY_THRESHOLD

    def _contains_wake_word(self, audio_path):
        if not self._model: return False
        result = self._model.transcribe(audio_path, language="ko", fp16=False)
        text = result["text"].strip().replace(" ", "")
        return any(w.replace(" ", "") in text for w in self.wake_words)

    def listen_and_trigger(self):
        # Colab 환경 주의: 마이크 장치가 없을 경우 여기서 에러 발생
        try:
            print(f"🎤 [대기 중] '{self.wake_words[0]}'라고 말해보세요...")
            chunk_samples = int(1.5 * 16000)

            while True:
                chunk = sd.rec(chunk_samples, samplerate=16000, channels=1, dtype="float32")
                sd.wait()
                audio = chunk.flatten()

                if not self._is_speech(audio): continue

                tmp_path = "temp_wake.wav"
                sf.write(tmp_path, audio, 16000)

                if self.use_whisper and not self._contains_wake_word(tmp_path):
                    continue

                print("🔔 [감지] '헬로비'를 인식했습니다! 말씀을 시작하세요.")
                return self._record_after_wake()
        except Exception as e:
            print(f"⚠️ 마이크 인식 불가(환경 문제): {e}")
            return None

    def _record_after_wake(self):
        print("🔴 [녹음 중] 7초간 말씀해 주세요...")
        audio = sd.rec(int(7 * 16000), samplerate=16000, channels=1, dtype="float32")
        sd.wait()
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        sf.write(tmp.name, audio.flatten(), 16000)
        return tmp.name

print("✅ WakeWord 모듈 로드 완료 (Step 0)")

## Step 4 — A팀 STT 전처리 함수

In [ ]:
import torch
import soundfile as sf
import librosa
import numpy as np
import tempfile
import re

# 1. Silero-VAD 모델 로드
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    trust_repo=True
)
(get_speech_timestamps, *_) = utils

# 2. 전처리용 상수 및 확장된 의료 보정 사전 (50개 이상 패턴)
SAMPLE_RATE = 16000
WAKE_WORDS = ['헬로비야', '헬로비이', '헬로 비', '헬로비']
FILLER_PATTERN = re.compile(r'(?<!\w)(어+~*|음+~*|에+~*|그+~*|뭐+~*|저+~*|아+~*)(?=\s|$)(?!\w)')

In [ ]:
# [고도화] 5개 카테고리별 의료 용어 보정 사전
# 노인 음성 인식 오류 패턴(받침 뭉개짐, 유사 발음 등)을 모두 반영했습니다.
MEDICAL_CORRECTIONS = {
    # 1. 진료과 (Department)
    "정형외가": "정형외과", "정형외꽈": "정형외과", "정형외와": "정형외과", "정영외과": "정형외과",
    "이비인후가": "이비인후과", "이비인호과": "이비인후과", "이비인우과": "이비인후과",
    "소화기가": "소화기내과", "피부가": "피부과", "피부부가": "피부과", "안과가": "안과",
    "내과가": "내과", "신경가": "신경과", "신경내가": "신경내과", "산부인가": "산부인과",
    "흉부외가": "흉부외과", "비뇨기가": "비뇨의학과", "재활의학가": "재활의학과", "가정의학가": "가정의학과",

    # 2. 증상 및 신체 부위 (Symptom & Body Part)
    "무릅": "무릎", "어꺠": "어깨", "머리아포": "두통", "배아포": "복통", "울렁거려": "구역질",
    "체했어": "소화불량", "소화안돼": "소화불량", "오심이": "오심", "구통이": "구토",
    "기침이": "기침", "가래가": "가래", "콧물나": "콧물", "숨차": "호흡곤란", "붓기": "부종",
    "쑤셔": "통증", "결려": "통증", "욱신거려": "통증", "띵해": "두통", "가슴답답": "흉통",

    # 3. 약 (Medication)
    "혈압야": "혈압약", "혈압아": "혈압약", "혈압박": "혈압약", "당뇨야": "당뇨약", "당뇨약이": "당뇨약",
    "감기야": "감기약", "감기박": "감기약", "수면야": "수면약", "타이래놀": "타이레놀",
    "진통제가": "진통제", "소염제가": "소염제", "항생제가": "항생제",

    # 4. 검사 및 처치 (Examination)
    "엑스레이": "X-ray", "엑스래이": "X-ray", "엠알아이": "MRI", "씨티": "CT",
    "피검사": "혈액검사", "혈액검사가": "혈액검사", "소변검사가": "소변검사", "초음파가": "초음파",

    # 5. 병명 (Disease)
    "고혈암": "고혈압", "당뇨병이": "당뇨병", "골다골증": "골다공증", "관절염이": "관절염",
    "치매가": "치매", "뇌경색이": "뇌경색", "뇌출혈이": "뇌출혈", "심근경새": "심근경색", "심근경섹": "심근경색"
}

In [ ]:
# 3. 보조 함수들
def remove_silence(audio_path: str) -> str:
    """VAD 필터로 침묵 제거"""
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    wav = torch.from_numpy(audio)
    speech_timestamps = get_speech_timestamps(wav, vad_model, sampling_rate=SAMPLE_RATE, threshold=0.4)
    if not speech_timestamps: return audio_path
    speech_audio = torch.cat([wav[ts["start"]:ts["end"]] for ts in speech_timestamps])
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, speech_audio.numpy(), SAMPLE_RATE)
    return tmp.name

def preprocess_text(raw_text: str) -> str:
    """A팀 고도화 로직: 호출어 제거 -> 간투어 필터링 -> 용어 교정 -> 중복 제거"""
    text = raw_text
    # 호출어 및 '야' 제거
    for ww in WAKE_WORDS:
        text = text.replace(ww, '')
    text = re.sub(r'^[야아아]+\s*', '', text).strip()

    # [A팀 핵심] 정규표현식을 이용한 간투어 제거
    text = FILLER_PATTERN.sub('', text).strip()

    # 의료 용어 보정 사전 적용
    for wrong, correct in MEDICAL_CORRECTIONS.items():
        text = text.replace(wrong, correct)

    # 연속된 중복 단어 제거 및 공백 정규화
    words = text.split()
    deduped = [w for i, w in enumerate(words) if i == 0 or w != words[i-1]]
    text = ' '.join(deduped)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# 4. 최종 통합 STT 파이프라인
def stt_pipeline(audio_path: str = None, raw_text: str = None, confidence: float = 0.94) -> dict:
    if audio_path:
        clean_audio_path = remove_silence(audio_path)
        audio_array, _ = librosa.load(clean_audio_path, sr=16000)
        input_features = stt_processor(audio_array, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
        predicted_ids = stt_model.generate(input_features)
        raw_text = stt_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        confidence = 0.90

    clean_text = preprocess_text(raw_text)
    return {'text': clean_text, 'raw_text': raw_text, 'confidence': confidence, 'language': 'ko'}

print('✅ A팀 STT + VAD + 고도화 보정 사전 통합 완료!')

### Step 5 - B팀 파인튜닝 LLM 모델

In [ ]:
import json
import re
import torch
from unsloth import FastLanguageModel



# 2. 전역 데이터 및 시니어 맞춤형 질문 사전
EMERGENCY_KEYWORDS = [
    '숨이 안 쉬어', '가슴이 너무 아프', '의식이 없', '쓰러', '피를 토',
    '말이 어눌', '입이 돌아', '한쪽이 마비', '갑자기 안 보여'
]

FOLLOWUP_QUESTIONS = {
    '무릎': '무릎이 많이 아프시군요. 혹시 걷기가 많이 힘드신가요?',
    '허리': '허리가 아프시군요. 혹시 허리를 펴거나 숙이기가 어려우신가요?',
    '어깨': '어깨가 불편하시군요. 팔을 위로 올리기가 힘드신 상태인가요?',
    '머리': '머리가 아프시군요. 갑자기 핑 돌거나 망치로 맞은 듯이 아픈가요?',
    '배': '배가 아프시군요. 속이 메스껍거나 콕콕 찌르는 느낌이 드세요?',
    '가슴': '가슴이 답답하시군요. 숨을 쉬기가 벅차거나 조이는 느낌인가요?',
}

# 3. 고도화된 답변 생성 함수 (The Warm Heart Engine)
def generate_answer(query: str, context: str, confidence: float = 0.85, entities: dict = None) -> dict:
    body_part = entities.get('body_part') if entities else "해당"

    # 시스템 프롬프트: 페르소나 및 맥락 지시
    system_prompt = f"당신은 어르신을 지극정성으로 모시는 다정한 의료 AI '헬로비'입니다."

    # 유저 메시지 구성
    user_msg = f"""현재 어르신은 '{body_part}' 부위가 불편하다고 하셨습니다.
아래 [참고 정보]를 바탕으로 어르신의 질문에 답변해 주세요.

### [참고 정보]
{context}

### [어르신의 질문]
{query}

### [출력 규칙]
1. 첫 문장은 무조건 어르신의 통증에 대해 걱정해주는 따뜻한 공감으로 시작하세요.
2. 가장 가까운 병원 한군데의 이름과 거리를 구체적으로 언급하고 조심히 다녀오시라고 다정하게 인사하세요.
3. 답변은 3~4문장 이내로 작성하세요."""

    try:
        # Groq API를 통한 답변 생성
        response = llm.invoke([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}
        ])

        return {'answer': response.content.strip()}

    except Exception as e:
        print(f"❌ Groq Answer Error: {e}")
        return {'answer': "어르신, 잠시 정보를 정리하는 데 시간이 조금 걸리네요. 다시 한번 말씀해 주시겠어요?"}

# 4. 의도 분류 및 엔티티 추출 (Local Inference 기반)
def local_inference(prompt: str, system_msg: str = "", max_tokens: int = 256) -> str:
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": prompt})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        attention_mask=(inputs != tokenizer.pad_token_id).long(),
        max_new_tokens=max_tokens,
        temperature=0.1, # 분류는 정확해야 하므로 낮게 설정
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

import re

# 2. 고도화된 의도 분류기 (Groq 전용)
def classify_intent(text: str) -> dict:
    clean_text = text.strip().replace(" ", "")

    # [1순위] 응급 상황 판별 (로컬 키워드 매칭 유지)
    if any(kw.replace(" ", "") in clean_text for kw in EMERGENCY_KEYWORDS):
        return {'intent': 'emergency', 'confidence': 1.0}

    # [2순위] LLM을 활용한 정교한 의도 분류
    try:
        prompt = (
            f"문장: '{text}'\n"
            "위 문장의 의도를 다음 4개 중 하나로만 대답하세요: "
            "[symptom_inquiry, hospital_search, medication_info, emergency]"
        )

        # 분류는 정확해야 하므로 낮은 temperature 적용
        ans = llm.invoke(prompt).content.lower()

        for target in ['emergency', 'medication_info', 'hospital_search', 'symptom_inquiry']:
            if target in ans:
                return {'intent': target, 'confidence': 0.95}

        return {'intent': 'symptom_inquiry', 'confidence': 0.70}

    except Exception as e:
        print(f"❌ Groq Intent Error: {e}")
        return {'intent': 'symptom_inquiry', 'confidence': 0.50}

def extract_entities(text: str) -> dict:
    try:
        prompt = f"문장: '{text}'\n증상과 신체부위를 JSON 형식으로만 추출하세요. 예: {{\"symptom\": \"통증\", \"body_part\": \"무릎\"}}"
        ans = llm.invoke(prompt).content

        match = re.search(r'\{.*?\}', ans, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {'symptom': None, 'body_part': None}
    except Exception:
        return {'symptom': None, 'body_part': None}

# 5. 다중턴 및 통합 대화 관리 (The Multi-turn Logic)
def chat_with_followup(user_input: str, session_id: str = 'default') -> dict:
    global conversation_state
    if session_id not in conversation_state:
        conversation_state[session_id] = {'step': 1, 'body_part': None}
    state = conversation_state[session_id]

    # 의도 파악
    intent_res = classify_intent(user_input)

    # [Case 1] 응급 상황
    if intent_res['intent'] == 'emergency':
        conversation_state[session_id] = {'step': 1, 'body_part': None}
        return {'answer': '어이구 어르신, 지금 많이 위험하실 수 있어요! 바로 119에 전화하시는 게 좋겠어요.', 'intent': 'emergency', 'ready_for_c': True,
                'output_for_c': {'intent': 'emergency', 'entities': {}, 'query': user_input, 'confidence': 0.99}}

    # [Case 2] 병원 검색 / 복약 정보 (바로 C단계로 이동)
    if intent_res['intent'] in ['medication_info', 'hospital_search']:
        entities = extract_entities(user_input)
        return {'answer': None, 'intent': intent_res['intent'], 'ready_for_c': True,
                'output_for_c': {'intent': intent_res['intent'], 'entities': entities, 'query': user_input, 'confidence': 0.90}}

    # [Case 3] 증상 문의 (다중턴 적용)
    if state['step'] == 1:
        for part, question in FOLLOWUP_QUESTIONS.items():
            if part in user_input:
                state['body_part'] = part
                state['step'] = 2
                return {'answer': question, 'intent': 'symptom_inquiry', 'ready_for_c': False, 'output_for_c': None}

        # 신체부위 언급이 없으면 바로 분석
        entities = extract_entities(user_input)
        return {'answer': None, 'intent': 'symptom_inquiry', 'ready_for_c': True,
                'output_for_c': {'intent': 'symptom_inquiry', 'entities': entities, 'query': user_input, 'confidence': 0.85}}

    # [Case 4] 다중턴 답변 수집 단계
    if state['step'] == 2:
        body_part = state['body_part']
        conversation_state[session_id] = {'step': 1, 'body_part': None} # 세션 초기화
        entities = {'symptom': f"{body_part} 통증 및 {user_input}", 'body_part': body_part}
        return {'answer': None, 'intent': 'symptom_inquiry', 'ready_for_c': True,
                'output_for_c': {'intent': 'symptom_inquiry', 'entities': entities, 'query': f"{body_part} 아픔. {user_input}", 'confidence': 0.91}}

print("✅ [최종] 헬로비 통합 두뇌 모듈 로드 완료!")

## Step 6 — C팀 RAG + 병원검색 + 응급판단 + Tool Router

In [ ]:
import torch
import gc

# 찌꺼기 메모리 강제 수거 및 GPU 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

In [ ]:

# 1) 임시 폴더에 압축 해제
!rm -rf /tmp/chroma_restore
!mkdir -p /tmp/chroma_restore
!unzip -o "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db.zip" -d /tmp/chroma_restore

# 2) 대상 db 폴더 정리 후 복사
!rm -rf "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db"
!mkdir -p "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG"
!cp -r /tmp/chroma_restore/db "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/"

In [ ]:
!ls -lh "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db/chroma.sqlite3"
!unzip -l "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db.zip" | sed -n '1,40p'

In [ ]:
# 1) 런타임 로컬에 확실히 복원
!rm -rf /tmp/chroma_restore
!mkdir -p /tmp/chroma_restore
!unzip -o "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db.zip" -d /tmp/chroma_restore

# 2) 로컬 복원본으로 count 확인 (핵심)
import chromadb
client = chromadb.PersistentClient(path="/tmp/chroma_restore/db")
col = client.get_or_create_collection("medical_knowledge", metadata={"hnsw:space":"cosine"})
print("tmp count:", col.count())

In [ ]:
# Colab에서는 복원본(/tmp) 경로를 우선 사용
if os.path.exists("/tmp/chroma_restore/db/chroma.sqlite3"):
    DB_PATH = "/tmp/chroma_restore/db"
elif os.path.exists("/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db"):
    DB_PATH = "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db"
else:
    DB_PATH = r"C:\Users\olivi\Desktop\LGHelloDoctor\LGHelloDoctor\RAG\db"

In [ ]:
import os
import chromadb
import requests
import xml.etree.ElementTree as ET
from sentence_transformers import SentenceTransformer
from numpy import dot
from numpy.linalg import norm

# =========================
# ChromaDB + 임베딩 모델
# =========================
# 우선순위:
# 1) /tmp 복원본 (가장 안정적)
# 2) Drive db
# 3) 로컬 Windows 경로(코랩에서는 보통 안 씀)
if os.path.exists("/tmp/chroma_restore/db/chroma.sqlite3"):
    DB_PATH = "/tmp/chroma_restore/db"
elif os.path.exists("/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db/chroma.sqlite3"):
    DB_PATH = "/content/drive/MyDrive/LGHellovision/LG_HelloDoctor/RAG/db"
else:
    DB_PATH = r"C:\Users\olivi\Desktop\LGHelloDoctor\LGHelloDoctor\RAG\db"

os.makedirs(DB_PATH, exist_ok=True)

embed_model = SentenceTransformer("jhgan/ko-sroberta-multitask")
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(
    "medical_knowledge",
    metadata={"hnsw:space": "cosine"},
)

print("DB_PATH:", DB_PATH)
print(f"ChromaDB 문서 수(초기): {collection.count()}개")

if collection.count() == 0:
    raise RuntimeError(
        "ChromaDB가 0개입니다. 먼저 /tmp/chroma_restore/db 로 unzip 되었는지 확인하세요."
    )

# =========================
# RAG
# =========================
QUERY_REWRITE_MAP = {
    "무릎": "무릎관절염 정형외과 관절 통증 진료",
    "허리": "허리디스크 정형외과 척추 통증 진료",
    "어깨": "오십견 정형외과 어깨 통증 진료",
    "머리": "편두통 신경과 두통 진료",
    "배": "위염 소화불량 소화기내과 진료",
    "가슴": "심근경색 심장내과 흉통 진료",
    "혈압약": "혈압약 복용 방법 주의사항",
    "당뇨약": "당뇨약 복용 방법 주의사항",
}

def query_rewrite(query: str) -> str:
    for kw, rewritten in QUERY_REWRITE_MAP.items():
        if kw in query:
            return rewritten
    return query

def full_rag_pipeline(query: str) -> str:
    if collection.count() == 0: return ""

    rewritten = query_rewrite(query)
    q_emb = embed_model.encode([rewritten]).tolist()

    # 유사도 거리(distance)를 함께 가져옴
    vec_res = collection.query(query_embeddings=q_emb, n_results=3)

    filtered_docs = []
    if vec_res and 'distances' in vec_res:
        # 코사인 거리 기준 0.45 이하(유사도 약 0.55 이상)만 채택
        for doc, dist in zip(vec_res['documents'][0], vec_res['distances'][0]):
            if dist <= 0.45:
                filtered_docs.append(doc)

    if not filtered_docs:
        # 검색 결과가 기준 미달이면 일반적인 안내만 포함
        return "관련된 구체적인 질환 정보를 찾지 못했습니다. 일반적인 진료 지침을 참고하세요."

    return " ".join(filtered_docs)

    q_emb2 = embed_model.encode([query])
    d_embs = embed_model.encode(combined)
    scores = [
        (dot(q_emb2[0], d_emb) / (norm(q_emb2[0]) * norm(d_emb) + 1e-9), combined[i])
        for i, d_emb in enumerate(d_embs)
    ]
    scores.sort(reverse=True)
    return " ".join([d for _, d in scores[:3]])

# =========================
# 병원 검색
# =========================
SYMPTOM_DEPT_MAP = {
    "무릎": ("정형외과", "05"), "허리": ("정형외과", "05"), "어깨": ("정형외과", "05"),
    "눈": ("안과", "12"), "귀": ("이비인후과", "13"), "코": ("이비인후과", "13"),
    "피부": ("피부과", "14"), "소변": ("비뇨의학과", "15"), "머리": ("신경과", "02"),
    "가슴": ("심장내과", "01"), "배": ("소화기내과", "01"), "혈압": ("내과", "01"),
}

def search_kakao(dept_name, lat, lng):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": dept_name, "x": lng, "y": lat, "radius": 3000, "category_group_code": "HP8", "size": 5}

    try:
        res = requests.get(url, headers=headers, params=params, timeout=5)
        docs = res.json().get("documents", [])
        results = []
        for p in docs:
            navi_link = f"https://map.kakao.com/link/to/{p['place_name']},{p['y']},{p['x']}"
            results.append({
                "name": p["place_name"],
                "address": p.get("road_address_name", ""),
                "phone": p.get("phone", ""),
                "distance": int(p.get("distance", 999999)),
                "navi_url": navi_link,
                "lat": p["y"],
                "lng": p["x"],
            })
        return results
    except Exception:
        return []

def search_hospital(symptom_text, lat=37.5012, lng=127.0396):
    dept_name = "내과"
    for symptom, (name, _) in SYMPTOM_DEPT_MAP.items():
        if symptom in symptom_text:
            dept_name = name
            break
    hospitals = sorted(
        [h for h in search_kakao(dept_name, lat, lng) if h.get("phone")],
        key=lambda x: x["distance"]
    )
    return {"department": dept_name, "nearby": hospitals[:3]}

# =========================
# 응급 판단
# =========================
EMERGENCY_SCORES = {
    "숨이 안 쉬어": 100, "의식이 없": 100, "피를 토": 90,
    "가슴이 너무 아프": 90, "한쪽이 마비": 90, "말이 어눌": 85,
    "쓰러": 80, "혈압이 200": 80, "식은땀": 30, "가슴이 아파": 40,
}

def emergency_check(text):
    total, matched = 0, []
    for kw, score in EMERGENCY_SCORES.items():
        if kw in text:
            total += score
            matched.append(kw)
    if len(matched) >= 2:
        total = min(total * 1.2, 100)
    if total >= 70:
        return {"is_emergency": True, "severity": "HIGH", "score": round(total), "action": "지금 바로 119에 전화해 주세요."}
    if total >= 40:
        return {"is_emergency": True, "severity": "MEDIUM", "score": round(total), "action": "응급실에 가보시는 게 좋을 것 같아요."}
    return {"is_emergency": False, "severity": "LOW", "score": round(total), "action": None}

# =========================
# Tool Router
# =========================
def tool_router(output_from_B, lat=37.5012, lng=127.0396):
    intent = output_from_B.get("intent", "symptom_inquiry")
    query = output_from_B.get("query", "")
    result = {"intent": intent, "rag_context": None, "hospitals": None, "emergency": None}

    emerg = emergency_check(query)
    if emerg["is_emergency"]:
        result["emergency"] = emerg
        if emerg["severity"] == "HIGH":
            return result

    if intent == "symptom_inquiry":
        result["rag_context"] = full_rag_pipeline(query)
        result["hospitals"] = search_hospital(query, lat, lng)
    elif intent == "medication_info":
        result["rag_context"] = full_rag_pipeline(query)
    elif intent == "hospital_search":
        result["hospitals"] = search_hospital(query, lat, lng)

    return result

print("C팀 함수 로드 완료!")

In [ ]:
print('\n' + '='*70)
print('C팀 RAG 파이프라인 — B팀 출력으로 종단간 테스트')
print('='*70 + '\n')

# B팀 출력 모의 데이터
b_output_samples = [
    {
        'name': '무릎 통증 (멀티턴)',
        'data': {
            'intent': 'symptom_inquiry',
            'query': '무릎이 아파요. 많이 힘들어요',
            'entities': {'symptom': '무릎 통증', 'body_part': '무릎'},
            'severity': None,
            'turn1_text': '무릎이 아파요',
            'turn2_text': '많이 힘들고 걷기 어려워요'
        }
    },
    {
        'name': '응급 상황 (HIGH severity)',
        'data': {
            'intent': 'emergency',
            'query': '가슴이 아프고 숨이 안 쉬어져요',
            'entities': {'symptom': '흉통 호흡곤란', 'body_part': '가슴'},
            'severity': 'HIGH',
            'turn1_text': '가슴이 아프고 숨이 안 쉬어져요',
            'turn2_text': None
        }
    },
    {
        'name': '복약 정보',
        'data': {
            'intent': 'medication_info',
            'query': '혈압약이랑 감기약 같이 먹어도 되나요',
            'entities': {'symptom': None, 'body_part': None},
            'severity': None,
            'turn1_text': '혈압약이랑 감기약 같이 먹어도 되나요',
            'turn2_text': None
        }
    },
]

for sample in b_output_samples:
    print(f"[{"—" * 50}]")
    print(f"📋 {sample['name']}")
    print(f"{"—" * 50}")

    b_data = sample['data']
    print(f"B팀 input: intent={b_data['intent']}, severity={b_data['severity']}")
    print(f"           query: {b_data['query'][:50]}...")

    try:
        # C팀 tool_router 호출
        result = tool_router(b_data)

        print(f"\n✓ C팀 라우터 처리 완료:")
        print(f"  Intent:    {result['intent']}")

        if result['emergency']:
            print(f"  🚨 응급:   {result['emergency']['action']}")

        if result['rag_context']:
            ctx = result['rag_context'][:80].replace('\n', ' ')
            print(f"  📚 RAG:    {ctx}...")

        if result['hospitals']:
            dept = result['hospitals'].get('department', '?')
            nearby_count = len(result['hospitals'].get('nearby', []))
            print(f"  🏥 병원:   {dept} ({nearby_count}개 찾음)")

    except Exception as e:
        print(f"✗ 오류: {e}")

    print()

print('='*70)
print('통합 테스트 완료!')
print('='*70)


## Step 7 — D팀 응답 포맷터

In [ ]:
# =============================================================================
# Step 7 — D팀 응답 포맷터 & TTS (음성 답변 생성)
# =============================================================================
!pip install gTTS -q

from gtts import gTTS
import os
import re
import tempfile

# 의료법 및 안전 가이드라인 필터링 단어
FORBIDDEN_WORDS = [
    '예후', '처방전', '투약', '병변', '진단',
    '확정', '완치', '확신', '치료', '부작용'
]

def format_response(raw_answer: str, is_emergency: bool = False) -> str:
    if is_emergency:
        return '지금 바로 119에 전화해 주세요. 매우 위험한 상황일 수 있습니다.'
    if not raw_answer:
        return "죄송합니다. 다시 한번 말씀해 주시겠어요?"

    answer = raw_answer
    for word in FORBIDDEN_WORDS:
        answer = answer.replace(word, '')

    answer = re.sub(r'\s+', ' ', answer).strip()

    # 문장 단위 분리 후 6문장까지 허용 (병원 정보와 인사말 보존)
    sentences = re.split(r'([.!?])', answer)
    combined_sentences = []
    for i in range(0, len(sentences)-1, 2):
        s = sentences[i].strip() + sentences[i+1]
        if s: combined_sentences.append(s)

    # 잘리는 범위를 [:6]으로 확대
    final_text = ' '.join(combined_sentences[:6]).strip()
    return final_text if final_text else answer

def generate_tts(text: str) -> str:
    """
    텍스트를 음성으로 변환하여 임시 파일 경로를 반환합니다.
    시니어 사용자를 위해 한국어(ko) 설정을 적용합니다.
    """
    try:
        # gTTS 객체 생성 (천천히 읽기 옵션은 slow=False가 자연스러움)
        tts = gTTS(text=text, lang='ko')

        # 임시 파일 생성 및 저장
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        tts.save(tmp.name)
        return tmp.name
    except Exception as e:
        print(f"❌ TTS 생성 에러: {e}")
        return None

print('✅ D팀 포맷터 및 TTS 모듈 로드 완료!')

## Step 8 — 전체 파이프라인 함수

In [ ]:
def full_pipeline(raw_text: str, session_id: str = 'default', lat: float = 37.5012, lng: float = 127.0396) -> dict:
    """
    A(STT) → B(의도/다중턴) → C(RAG/병원/응급) → D(LLM/TTS) 통합 파이프라인
    """
    print(f'\n[시작] 입력 데이터: {raw_text}')
    print('-' * 60)

    # --- [A] 전처리 및 STT ---
    audio_extensions = ('.wav', '.mp3', '.m4a', '.flac', '.ogg')
    if isinstance(raw_text, str) and raw_text.lower().endswith(audio_extensions):
        stt_output = stt_pipeline(raw_text)
        text = stt_output['text']
    else:
        text = raw_text
    print(f'[A] 최종 인식 문장: {text}')

    # --- [B] 의도 분류 및 다중턴 로직 ---
    b_result = chat_with_followup(text, session_id)
    print(f'[B] 의도: {b_result["intent"]} / ready_for_c: {b_result["ready_for_c"]}')

    # 다중턴 질문(추가 질문)이 필요한 경우 즉시 반환
    if not b_result['ready_for_c']:
        answer = format_response(b_result['answer'])
        return {
            'answer': answer,
            'intent': b_result['intent'],
            'ready_for_c': False,  # 프론트엔드에게 "더 물어봐야 함"을 알림
            'hospitals': None,
            'emergency': None
        }

    # --- [C] 도구 활용 (RAG, 병원검색, 응급판단) ---
    output_from_B = b_result['output_for_c']
    c_result = tool_router(output_from_B, lat, lng)

    # 병원 리스트를 AI가 읽기 좋은 텍스트로 가공
    hospital_info_text = ""
    if c_result.get('hospitals') and c_result['hospitals'].get('nearby'):
        h_list = c_result['hospitals']['nearby']
        hospital_info_text = "\n[주변 추천 병원 목록]\n"
        for i, h in enumerate(h_list[:3]):
            walk_time = round(h['distance'] / 66.6)
            hospital_info_text += (
                f"{i+1}. {h['name']}: 거리 {h['distance']}m, 도보 약 {walk_time}분\n"
                f"   - 주소: {h['address']}\n"
                f"   - 전화: {h['phone']}\n"
            )

    # RAG 지식과 병원 정보를 하나로 병합
    rag_context = c_result.get('rag_context') or ""
    combined_context = f"{rag_context}\n{hospital_info_text}".strip()

    # [디버깅 로그] AI에게 전달되는 실제 데이터 재료 확인
    print("\n" + "="*20 + " [C] DEBUG: COMBINED CONTEXT " + "="*20)
    print(combined_context if combined_context else "제공할 컨텍스트 데이터 없음")
    print("="*65 + "\n")

    # --- [D] 최종 응답 및 TTS ---
    # 1. 응급 상황 우선 처리
    if c_result['emergency'] and c_result['emergency']['severity'] == 'HIGH':
        final_answer = format_response('', is_emergency=True)

    # 2. 일반 답변 생성
    else:
        if combined_context:
            # [중요 수정] entities 인자를 반드시 전달해야 맥락 유지가 됩니다.
            answer_result = generate_answer(
                text,
                context=combined_context,
                confidence=output_from_B.get('confidence', 0.85),
                entities=output_from_B.get('entities') # 이 부분이 추가되어야 함
            )
            raw_answer = answer_result['answer']
        else:
            raw_answer = b_result['answer'] or "죄송해요, 관련 정보를 찾지 못했습니다."

        final_answer = format_response(raw_answer)

    print(f'[D] 최종 답변: {final_answer}')
    print(f'[D] 음성 답변 생성 중...')
    audio_path = generate_tts(final_answer)

    return {
        'answer': final_answer,
        'audio_path': audio_path,
        'intent': b_result['intent'],
        'ready_for_c': True,  # 이 줄을 반드시 추가해야 프론트엔드 마이크가 꺼집니다.
        'hospitals': c_result.get('hospitals'),
        'emergency': c_result.get('emergency'),
    }

print('🚀 [최종 확인] 모든 파이프라인이 정석대로 통합되었습니다!')

In [ ]:
# '나 배고파' 시나리오 테스트 코드
print('=== 일상 대화 테스트: 배고픔 ===')

# 1. 파이프라인 실행
test_input = '나 배고파'
result = full_pipeline(test_input, session_id='test_hunger')

# 2. 주요 결과 출력
print(f"\n[최종 결과 확인]")
print(f"입력 문장: {test_input}")
print(f"분류된 의도: {result['intent']}")
print(f"헬로비의 답변: {result['answer']}")

if result.get('hospitals'):
    dept = result['hospitals'].get('department')
    print(f"추천된 진료과: {dept}")

## Step 9 — 시나리오 테스트

In [ ]:
# [검증용] Groq 기반 멀티턴 파이프라인 엔진
import json

def chat_with_followup(user_input: str, session_id: str = 'default') -> dict:
    global conversation_state

    # 새로운 세션이면 1단계로 시작
    if session_id not in conversation_state:
        conversation_state[session_id] = {'step': 1, 'body_part': None}

    state = conversation_state[session_id]

    # 의도 분류 호출 (Groq API 활용)
    intent_res = classify_intent(user_input)

    # [멀티턴 핵심 로직: 증상 문의]
    if intent_res['intent'] == 'symptom_inquiry':
        # 1단계: 신체 부위가 언급되었는지 확인
        if state['step'] == 1:
            for part, question in FOLLOWUP_QUESTIONS.items():
                if part in user_input:
                    state['body_part'] = part
                    state['step'] = 2 # 다음 단계를 위해 상태 변경
                    return {
                        'answer': question,
                        'intent': 'symptom_inquiry',
                        'ready_for_c': False, # C단계(RAG)로 가지 않고 질문을 던짐
                        'output_for_c': None
                    }

        # 2단계: 추가 답변을 받았거나, 부위 정보가 이미 있는 경우
        if state['step'] == 2:
            body_part = state['body_part']
            # 데이터 취합 후 세션 초기화
            conversation_state[session_id] = {'step': 1, 'body_part': None}
            entities = {'symptom': f"{body_part} 통증 및 {user_input}", 'body_part': body_part}
            return {
                'answer': None,
                'intent': 'symptom_inquiry',
                'ready_for_c': True, # 이제 정보가 충분하므로 C단계(RAG) 가동
                'output_for_c': {'intent': 'symptom_inquiry', 'entities': entities, 'query': user_input}
            }

    # 응급/복약/병원찾기는 멀티턴 없이 즉시 처리
    entities = extract_entities(user_input)
    return {
        'answer': None,
        'intent': intent_res['intent'],
        'ready_for_c': True,
        'output_for_c': {'intent': intent_res['intent'], 'entities': entities, 'query': user_input}
    }

In [ ]:
# 시나리오 A — 증상 (2턴) 테스트 수정본
print('=== 시나리오 A: 무릎 통증 ===')
r1 = full_pipeline('헬로비 머리 아파', 'scenario_A')
print(f"🤖 헬로비의 추가 질문: {r1['answer']}") # 이 줄을 추가해야 질문이 보입니다!

print('-' * 30)

r2 = full_pipeline('그냥 아파', 'scenario_A')
print(f"🤖 헬로비의 최종 답변: {r2['answer']}")
# 시나리오 B — 응급
print('=== 시나리오 B: 응급 ===')
full_pipeline('헬로비야 가슴이 아프고 숨이 안 쉬어져요', 'scenario_B')

print()

# 시나리오 C — 복약
print('=== 시나리오 C: 복약 ===')
full_pipeline('헬로비 혈압약이랑 감기약 같이 먹어도 되나요', 'scenario_C')



## Step 10 — FastAPI 서버 + ngrok -

In [ ]:
# =============================================================================
# [수정본] Step 10 — FastAPI 서버 + ngrok (지능적 음성 제어 지원)
# =============================================================================
!pip install nest-asyncio pyngrok -q

import nest_asyncio
import uvicorn
import tempfile
import os
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title='LG HelloDoctor API')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_credentials=True,
    allow_methods=['*'], allow_headers=['*'],
)

# 데이터 규격 정의 (ready_for_c 필드 추가)
class ChatRequest(BaseModel):
    text: str
    session_id: str = 'default'
    lat: float = 37.5012
    lng: float = 127.0396

class ChatResponse(BaseModel):
    answer: str
    intent: str
    hospitals: Optional[list] = None
    is_emergency: bool = False
    ready_for_c: bool  # 프론트엔드 마이크 제어의 핵심 플래그
    session_id: str

@app.get('/')
def root():
    return {'message': 'LG HelloDoctor API 정상 작동 중'}

# -----------------------------------------------------------------------------
# 1. 음성 파일 처리 API (/api/stt)
# -----------------------------------------------------------------------------
@app.post("/api/stt")
async def stt_endpoint(audio: UploadFile = File(...)):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        content = await audio.read()
        tmp.write(content)
        tmp_path = tmp.name

    try:
        stt_result = stt_pipeline(audio_path=tmp_path)
        return {
            "text": stt_result['text'],
            "raw_text": stt_result['raw_text'],
            "status": "success"
        }
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

# -----------------------------------------------------------------------------
# 2. 통합 채팅 API (/chat) - ready_for_c 로직 반영
# -----------------------------------------------------------------------------
@app.post('/chat', response_model=ChatResponse)
def chat(request: ChatRequest):
    # Step 8의 full_pipeline 호출
    result = full_pipeline(
        raw_text=request.text,
        session_id=request.session_id,
        lat=request.lat,
        lng=request.lng
    )

    hospitals = result.get('hospitals', {}).get('nearby', []) if result.get('hospitals') else []

    # [중요] result['ready_for_c']가 False면 "추가 질문 중", True면 "정보 제공 완료"
    return ChatResponse(
        answer=result['answer'],
        intent=result['intent'],
        hospitals=hospitals,
        is_emergency=result.get('is_emergency', result['intent'] == 'emergency'),
        ready_for_c=result.get('ready_for_c', True),
        session_id=request.session_id
    )

# ngrok 설정 및 서버 가동
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
print(f'\n🚀 외부 접속 URL: {tunnel.public_url}')
print(f'📋 API 문서 확인: {tunnel.public_url}/docs')

config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [3064]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🚀 외부 접속 URL: https://johanne-crystallographic-miguelina.ngrok-free.dev
📋 API 문서 확인: https://johanne-crystallographic-miguelina.ngrok-free.dev/docs


/tmp/ipykernel_3064/3928885135.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from inpu

INFO:     211.248.242.211:0 - "POST /api/stt HTTP/1.1" 200 OK

[시작] 입력 데이터: 머리 아파
------------------------------------------------------------
[A] 최종 인식 문장: 머리 아파
[B] 의도: symptom_inquiry / ready_for_c: False
INFO:     211.248.242.211:0 - "POST /chat HTTP/1.1" 200 OK


/tmp/ipykernel_3064/3928885135.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


...

INFO:     211.248.242.211:0 - "POST /api/stt HTTP/1.1" 200 OK
INFO:     211.248.242.211:0 - "OPTIONS /chat HTTP/1.1" 200 OK

[시작] 입력 데이터: 가능할 수 있게 하고 프로그램도 계속 많이 쌓아가고
------------------------------------------------------------
[A] 최종 인식 문장: 가능할 수 있게 하고 프로그램도 계속 많이 쌓아가고
[B] 의도: symptom_inquiry / ready_for_c: True

==================== [C] DEBUG: COMBINED CONTEXT ====================
관련된 구체적인 질환 정보를 찾지 못했습니다. 일반적인 진료 지침을 참고하세요.

[주변 추천 병원 목록]
1. 더영의원: 거리 129m, 도보 약 2분
   - 주소: 서울 종로구 종로 74
   - 전화: 02-722-1854
2. 센터원지앤이내과의원: 거리 276m, 도보 약 4분
   - 주소: 서울 중구 을지로5길 26
   - 전화: 02-6030-8966
3. 사랑담은내과의원: 거리 421m, 도보 약 6분
   - 주소: 서울 중구 무교로 32
   - 전화: 02-3789-8575

[D] 최종 답변: 어르신, 머리 부위가 불편하신 것 같아 정말 걱정되네요. 어르신의 건강이 제일 중요하니까 가능한 한 조심하시고, 가까운 병원을 방문해 보시는 것이 좋을 것 같아요. 더영의원은 여기서 걸어서 약 2분 거리인 129m 떨어져 있으니, 조심히 다녀오세요. 어르신의 건강이 곧 나아지시기를 바랍니다.
[D] 음성 답변 생성 중...
INFO:     211.248.242.211:0 - "POST /chat HTTP/1.1" 200 OK


/tmp/ipykernel_3064/3928885135.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


INFO:     211.248.242.211:0 - "POST /api/stt HTTP/1.1" 200 OK

[시작] 입력 데이터: 머리 아파
------------------------------------------------------------
[A] 최종 인식 문장: 머리 아파
[B] 의도: symptom_inquiry / ready_for_c: False
INFO:     211.248.242.211:0 - "POST /chat HTTP/1.1" 200 OK


/tmp/ipykernel_3064/3928885135.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


INFO:     211.248.242.211:0 - "POST /api/stt HTTP/1.1" 200 OK

[시작] 입력 데이터: 머리가 깨질듯이 아파
------------------------------------------------------------
[A] 최종 인식 문장: 머리가 깨질듯이 아파
[B] 의도: symptom_inquiry / ready_for_c: True

==================== [C] DEBUG: COMBINED CONTEXT ====================
무릎, 허리, 어깨 통증이 지속되면 정형외과 진료를 통해 근골격계 질환 여부를 확인하는 것이 좋습니다. 무릎, 허리, 어깨 통증이 지속되면 정형외과 진료를 통해 근골격계 질환 여부를 확인하는 것이 좋습니다. 뇌졸중은 뇌혈관 문제로 갑작스러운 편측 마비, 발음 이상, 의식 저하가 나타날 수 있는 응급 질환입니다.

[주변 추천 병원 목록]
1. 백명기의원: 거리 906m, 도보 약 14분
   - 주소: 서울 중구 명동8길 47
   - 전화: 02-775-9238
2. 연세퍼스티어내과신경과의원: 거리 1490m, 도보 약 22분
   - 주소: 서울 중구 세종대로 23
   - 전화: 02-777-7555
3. 서울대학교병원 신경과: 거리 1719m, 도보 약 26분
   - 주소: 서울 종로구 대학로 101
   - 전화: 1588-5700

[D] 최종 답변: 어르신, 머리가 깨질듯이 아파하신다니 정말 걱정되네요. 어르신의 통증이 빨리 사라지길 바래요. 백명기의원은 여기서 도보 약 14분 거리입니다. 백명기의원에 조심히 다녀오시고, 어르신의 건강이 곧 나아지시길 바랍니다.
[D] 음성 답변 생성 중...
INFO:     211.248.242.211:0 - "POST /chat HTTP/1.1" 200 OK
